In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

train_df = pd.read_csv('kaggle_dataset/train.csv')
test_df = pd.read_csv('kaggle_dataset/test.csv')

def clean_data_pro(df, is_train=False, brand_freq=None):
    df = df.copy()
    
    if 'milage' in df.columns:
        df['milage'] = df['milage'].astype(str).str.replace(',', '').str.replace(' mi.', '').astype(float)
        df['milage_sq'] = df['milage'] ** 2
        df['milage_sqrt'] = np.sqrt(df['milage'])
        
    if 'model_year' in df.columns:
        df['vehicle_age'] = 2026 - df['model_year']
        df['vehicle_age_sq'] = df['vehicle_age'] ** 2
        
    def extract_hp(s):
        match = re.search(r'(\d+\.?\d*)HP', str(s))
        return float(match.group(1)) if match else np.nan

    def extract_liters(s):
        match = re.search(r'(\d+\.?\d*)L', str(s))
        return float(match.group(1)) if match else np.nan
        
    def extract_cylinders(s):
        match = re.search(r'(\d+) Cylinder', str(s))
        if match: return float(match.group(1))
        s_str = str(s)
        if 'V6' in s_str or 'Flat 6' in s_str: return 6.0
        if 'V8' in s_str: return 8.0
        if 'V10' in s_str: return 10.0
        if 'I4' in s_str: return 4.0
        return np.nan
        
    if 'engine' in df.columns:
        df['horsepower'] = df['engine'].apply(extract_hp)
        df['engine_liters'] = df['engine'].apply(extract_liters)
        df['cylinders'] = df['engine'].apply(extract_cylinders)
        
    if 'accident' in df.columns:
        df['has_accident'] = df['accident'].astype(str).apply(lambda x: 0 if 'None' in x else 1)
        
    fill_cols = ['horsepower', 'engine_liters', 'cylinders', 'milage_sq', 'milage_sqrt']
    for col in fill_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].mean())
            
    if 'brand' in df.columns:
        if is_train:
            brand_freq = df['brand'].value_counts(normalize=True)
        df['brand_freq'] = df['brand'].map(brand_freq).fillna(0)
            
    return df, brand_freq

train_clean, brand_freq_map = clean_data_pro(train_df, is_train=True)
test_clean, _ = clean_data_pro(test_df, is_train=False, brand_freq=brand_freq_map)

upper_limit = train_clean['price'].quantile(0.99)
train_clean = train_clean[train_clean['price'] < upper_limit]

categorical_cols = ['fuel_type', 'transmission', 'clean_title'] 
numeric_cols = ['vehicle_age', 'vehicle_age_sq', 'milage', 'milage_sq', 'milage_sqrt', 
                'horsepower', 'engine_liters', 'has_accident', 'brand_freq', 'cylinders']

combined_df = pd.concat([train_clean, test_clean], sort=False)
combined_encoded = pd.get_dummies(combined_df, columns=categorical_cols, drop_first=True)

n_train = train_clean.shape[0]
train_encoded = combined_encoded.iloc[:n_train].copy()
test_encoded = combined_encoded.iloc[n_train:].copy()

final_features = numeric_cols + [col for col in train_encoded.columns if col.startswith(tuple(categorical_cols))]

X = train_encoded[final_features]
y = train_encoded['price']
X_test_final = test_encoded[final_features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test_final)

X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
model_val = Ridge(alpha=15.0)
model_val.fit(X_train, y_train)
print(f"Validation R2 Score (Expected > 0.58): {r2_score(y_val, model_val.predict(X_val)):.5f}")

final_model = Ridge(alpha=15.0)
final_model.fit(X_scaled, y)

submission = pd.DataFrame({
    'id': test_df['id'],
    'price': final_model.predict(X_test_scaled)
})
submission.to_csv('submission.csv', index=False)
print("submission.csv created successfully!")

Validation R2 Score (Expected > 0.58): 0.57838
submission.csv created successfully!
